<a href="https://colab.research.google.com/github/pjm9814/Econometrics/blob/metrics_spring_2026/PS4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**PS4**
##**Peter Marshall**

In [3]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os

##**Functions, data-generating process**

In [5]:
np.random.seed(42)
TRUE_ATE = 3.0
N = 500
N_SIMS = 1000

# ── helpers ──────────────────────────────────────────────────────────────────

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def logistic_regression(X, y, max_iter=100, tol=1e-8):
    """Newton-Raphson logistic regression. X should include intercept column."""
    n, p = X.shape
    beta = np.zeros(p)
    for _ in range(max_iter):
        pi = sigmoid(X @ beta)
        pi = np.clip(pi, 1e-10, 1 - 1e-10)
        W = pi * (1 - pi)
        # gradient and Hessian
        grad = X.T @ (pi - y)
        H = X.T @ (W[:, None] * X)
        try:
            delta = np.linalg.solve(H, grad)
        except np.linalg.LinAlgError:
            break
        beta -= delta
        if np.max(np.abs(delta)) < tol:
            break
    return beta

def ols(X, y):
    """OLS via normal equations. X should include intercept column."""
    return np.linalg.lstsq(X, y, rcond=None)[0]

# ── data-generating process ───────────────────────────────────────────────────

def generate_data(n, ps_coef):
    """ps_coef: coefficient magnitude (same for X1 and -X2)."""
    X1 = np.random.randn(n)
    X2 = np.random.randn(n)
    propensity = sigmoid(ps_coef * X1 - ps_coef * X2)
    T = (np.random.uniform(size=n) < propensity).astype(float)
    eps = np.random.randn(n)
    Y = 2 + 3 * T + 1.5 * X1 - 1.0 * X2 + eps
    return X1, X2, T, Y, propensity

# ── estimators ───────────────────────────────────────────────────────────────

def ipw_estimator(T, Y, pi_hat):
    pi_hat = np.clip(pi_hat, 1e-6, 1 - 1e-6)
    return np.mean(T * Y / pi_hat - (1 - T) * Y / (1 - pi_hat))

def regression_estimator(T, Y, X1, X2,
                          ps_vars=('X1', 'X2'), outcome_vars=('X1', 'X2'),
                          X1_full=None, X2_full=None):
    """
    ps_vars / outcome_vars: which covariates to include in each model.
    X1_full / X2_full: full-sample covariates (same as X1, X2 here since n=N).
    """
    n = len(Y)
    # build outcome design matrix
    cols = [np.ones(n), T]
    if 'X1' in outcome_vars: cols.append(X1)
    if 'X2' in outcome_vars: cols.append(X2)
    Xout = np.column_stack(cols)
    beta = ols(Xout, Y)
    # beta: [intercept, beta_T, (beta1), (beta2)]
    beta_T = beta[1]

    # predict mu0, mu1 for full sample
    n_full = len(X1_full) if X1_full is not None else n
    X1f = X1_full if X1_full is not None else X1
    X2f = X2_full if X2_full is not None else X2

    cols0 = [np.ones(n_full), np.zeros(n_full)]
    cols1 = [np.ones(n_full), np.ones(n_full)]
    if 'X1' in outcome_vars:
        cols0.append(X1f); cols1.append(X1f)
    if 'X2' in outcome_vars:
        cols0.append(X2f); cols1.append(X2f)

    Xout0 = np.column_stack(cols0)
    Xout1 = np.column_stack(cols1)
    mu0 = Xout0 @ beta
    mu1 = Xout1 @ beta
    ate_reg = np.mean(mu1 - mu0)   # == beta_T in correctly-specified linear model
    return ate_reg, mu0, mu1

def fit_propensity(T, X1, X2, ps_vars=('X1', 'X2'),
                   X1_full=None, X2_full=None):
    """Fit propensity score model and return pi_hat for full sample."""
    n = len(T)
    cols = [np.ones(n)]
    if 'X1' in ps_vars: cols.append(X1)
    if 'X2' in ps_vars: cols.append(X2)
    Xps = np.column_stack(cols)
    beta_ps = logistic_regression(Xps, T)

    n_full = len(X1_full) if X1_full is not None else n
    X1f = X1_full if X1_full is not None else X1
    X2f = X2_full if X2_full is not None else X2

    cols_full = [np.ones(n_full)]
    if 'X1' in ps_vars: cols_full.append(X1f)
    if 'X2' in ps_vars: cols_full.append(X2f)
    Xps_full = np.column_stack(cols_full)
    return sigmoid(Xps_full @ beta_ps)

def aipw_estimator(T, Y, pi_hat, mu0, mu1):
    pi_hat = np.clip(pi_hat, 1e-6, 1 - 1e-6)
    term1 = T * (Y - mu1) / pi_hat + mu1
    term2 = (1 - T) * (Y - mu0) / (1 - pi_hat) + mu0
    return np.mean(term1 - term2)

# ── single simulation run ─────────────────────────────────────────────────────

def run_one_sim(ps_coef,
                ps_vars=('X1', 'X2'),
                outcome_vars=('X1', 'X2')):
    X1, X2, T, Y, _ = generate_data(N, ps_coef)

    # propensity model
    pi_hat = fit_propensity(T, X1, X2, ps_vars=ps_vars,
                            X1_full=X1, X2_full=X2)

    # outcome model
    ate_reg, mu0, mu1 = regression_estimator(T, Y, X1, X2,
                                              outcome_vars=outcome_vars,
                                              X1_full=X1, X2_full=X2)

    ate_ipw  = ipw_estimator(T, Y, pi_hat)
    ate_aipw = aipw_estimator(T, Y, pi_hat, mu0, mu1)

    return ate_ipw, ate_reg, ate_aipw

# ── run many simulations ──────────────────────────────────────────────────────

def run_simulations(ps_coef,
                    ps_vars=('X1', 'X2'),
                    outcome_vars=('X1', 'X2'),
                    n_sims=N_SIMS):
    results = np.array([run_one_sim(ps_coef, ps_vars, outcome_vars)
                        for _ in range(n_sims)])
    return results   # shape (n_sims, 3)  cols: IPW, Reg, AIPW

def compute_metrics(results):
    """Returns dict with bias, variance, mse for each estimator."""
    names = ['IPW', 'Regression', 'AIPW']
    metrics = {}
    for i, name in enumerate(names):
        est = results[:, i]
        bias = np.mean(est) - TRUE_ATE
        var  = np.var(est, ddof=1)
        mse  = np.mean((est - TRUE_ATE) ** 2)
        metrics[name] = {'Bias': bias, 'Variance': var, 'MSE': mse}
    return metrics

def print_table(metrics, label):
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    print(f"{'Estimator':<15} {'Bias':>10} {'Variance':>12} {'MSE':>12}")
    print(f"{'-'*50}")
    for name, m in metrics.items():
        print(f"{name:<15} {m['Bias']:>10.5f} {m['Variance']:>12.6f} {m['MSE']:>12.6f}")


#**Section 1.1: Correct specification**

In [6]:

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1.1 — Correct specification, varying propensity coefficients
# ─────────────────────────────────────────────────────────────────────────────

ps_coefs = [0.25, 0.50, 0.75, 1.00]
results_11 = {}

print("\n\n" + "="*60)
print("  SECTION 1.1 — Correct Specification")
print("="*60)

for coef in ps_coefs:
    label = f"PS coef = {coef}"
    print(f"\nRunning 1000 simulations for {label}...")
    res = run_simulations(coef)
    results_11[coef] = res
    metrics = compute_metrics(res)
    print_table(metrics, label)

# ── PS density plots for Section 1.1 ─────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for idx, coef in enumerate(ps_coefs):
    ax = axes[idx]
    # generate one draw
    np.random.seed(0)
    X1, X2, T, Y, propensity = generate_data(N, coef)
    ps_treated = propensity[T == 1]
    ps_control = propensity[T == 0]

    # KDE
    def kde(data, bw=None):
        if bw is None:
            bw = 1.06 * np.std(data) * len(data) ** (-0.2)
        x = np.linspace(0, 1, 300)
        density = np.array([np.mean(np.exp(-0.5 * ((xi - data) / bw) ** 2)) / (bw * np.sqrt(2 * np.pi))
                            for xi in x])
        return x, density

    x_t, d_t = kde(ps_treated)
    x_c, d_c = kde(ps_control)

    ax.plot(x_t, d_t, color='steelblue', lw=2, label='Treated')
    ax.plot(x_c, d_c, color='tomato',    lw=2, label='Control')
    ax.fill_between(x_t, d_t, alpha=0.2, color='steelblue')
    ax.fill_between(x_c, d_c, alpha=0.2, color='tomato')
    ax.set_title(f'PS coef = {coef}', fontsize=13)
    ax.set_xlabel('True Propensity Score')
    ax.set_ylabel('Density')
    ax.legend()
    ax.set_xlim(0, 1)

fig.suptitle('Section 1.1: True Propensity Score Distributions\n(Treated vs Control)', fontsize=14, fontweight='bold')
plt.tight_layout()
fig_path_11 = os.path.join(OUTPUT_DIR, "ps4_section11_ps_density.png")
plt.savefig(fig_path_11, dpi=150, bbox_inches='tight')
plt.close()
print(f"\n[Saved PS density plot: {fig_path_11}]")

# ── Summary comparison table for Section 1.1 ─────────────────────────────────

print("\n\n" + "="*70)
print("  SECTION 1.1 — SUMMARY: MSE Comparison Across PS Coefficients")
print("="*70)
print(f"{'PS coef':<10}", end="")
for name in ['IPW', 'Regression', 'AIPW']:
    print(f"  {name+' Bias':>14} {name+' MSE':>12}", end="")
print()
print("-"*70)
for coef in ps_coefs:
    m = compute_metrics(results_11[coef])
    print(f"{coef:<10}", end="")
    for name in ['IPW', 'Regression', 'AIPW']:
        print(f"  {m[name]['Bias']:>14.5f} {m[name]['MSE']:>12.6f}", end="")
    print()




  SECTION 1.1 — Correct Specification

Running 1000 simulations for PS coef = 0.25...

  PS coef = 0.25
Estimator             Bias     Variance          MSE
--------------------------------------------------
IPW                0.00283     0.009074     0.009072
Regression         0.00152     0.008439     0.008432
AIPW               0.00158     0.008449     0.008443

Running 1000 simulations for PS coef = 0.5...

  PS coef = 0.5
Estimator             Bias     Variance          MSE
--------------------------------------------------
IPW               -0.00070     0.014685     0.014670
Regression         0.00188     0.008365     0.008360
AIPW               0.00186     0.008659     0.008654

Running 1000 simulations for PS coef = 0.75...

  PS coef = 0.75
Estimator             Bias     Variance          MSE
--------------------------------------------------
IPW               -0.00954     0.038667     0.038719
Regression        -0.00414     0.010333     0.010340
AIPW              -0.00561  

*Compare the above results you get. What pattern do you find (in terms of the relative performance of the three estimators)? What do you think is the reason for such change?*

**The three estimators all have very little bias -- given that the model is correctly specified, this is unsurprising. However, it is worth noting that as the PS coefficient grows (some propensity scores approach 0 or 1), the variance in IPW explodes because the weights (1/pi and 1/(1-pi)) become more extreme. We see this reflected in the large MSE for IPW, as compared to the other two estimators. The regression MSE is basically unchanged (which, again, makes sense given that it doesn't rely on propensity scores at all). The AIPW estimator sees a more moderate increase in MSE, since it does incorporate the propensity score weights, but its outcome model component makes it more stable (relative to IPW).**

#**Section 1.2: Misspecification**

In [7]:
print("\n\n" + "="*60)
print("  SECTION 1.2 — Misspecification (DGP coef = 0.25)")
print("="*60)

misspec_scenarios = {
    "Only PS misspecified\n(PS: X1 only, Outcome: X1+X2)":
        dict(ps_vars=('X1',), outcome_vars=('X1', 'X2')),
    "Only Outcome misspecified\n(PS: X1+X2, Outcome: X1 only)":
        dict(ps_vars=('X1', 'X2'), outcome_vars=('X1',)),
    "Both misspecified\n(PS: X1 only, Outcome: X1 only)":
        dict(ps_vars=('X1',), outcome_vars=('X1',)),
}

results_12 = {}
for label, kwargs in misspec_scenarios.items():
    short_label = label.split('\n')[0]
    print(f"\nRunning simulations for: {short_label}...")
    res = run_simulations(0.25, **kwargs)
    results_12[label] = res
    metrics = compute_metrics(res)
    print_table(metrics, label)

# ── Also run the correctly-specified baseline for comparison ──────────────────
print("\nRunning correctly-specified baseline for comparison...")
res_baseline = run_simulations(0.25)
results_12["Correctly specified\n(PS: X1+X2, Outcome: X1+X2)"] = res_baseline
metrics = compute_metrics(res_baseline)
print_table(metrics, "Correctly specified (baseline)")

# ─────────────────────────────────────────────────────────────────────────────
# SUMMARY FIGURE — Section 1.2 boxplots
# ─────────────────────────────────────────────────────────────────────────────

scenario_labels_short = [
    "Correct\nSpec",
    "PS\nMisspec",
    "Outcome\nMisspec",
    "Both\nMisspec",
]

all_12_results = [
    results_12["Correctly specified\n(PS: X1+X2, Outcome: X1+X2)"],
    results_12["Only PS misspecified\n(PS: X1 only, Outcome: X1+X2)"],
    results_12["Only Outcome misspecified\n(PS: X1+X2, Outcome: X1 only)"],
    results_12["Both misspecified\n(PS: X1 only, Outcome: X1 only)"],
]

estimator_colors = ['steelblue', 'darkorange', 'seagreen']
estimator_names  = ['IPW', 'Regression', 'AIPW']

fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=False)

for ei, (ename, ecolor) in enumerate(zip(estimator_names, estimator_colors)):
    ax = axes[ei]
    data_per_scenario = [r[:, ei] for r in all_12_results]
    bp = ax.boxplot(data_per_scenario,
                    labels=scenario_labels_short,
                    patch_artist=True,
                    medianprops=dict(color='black', lw=2))
    for patch in bp['boxes']:
        patch.set_facecolor(ecolor)
        patch.set_alpha(0.6)
    ax.axhline(TRUE_ATE, color='red', linestyle='--', lw=1.5, label='True ATE = 3')
    ax.set_title(f'{ename}', fontsize=13, fontweight='bold')
    ax.set_ylabel('Estimated ATE')
    ax.legend(fontsize=9)
    ax.tick_params(axis='x', labelsize=9)

fig.suptitle('Section 1.2: ATE Estimates Under Misspecification\n(1000 simulations, n=500)', fontsize=13, fontweight='bold')
plt.tight_layout()
fig_path_12 = os.path.join(OUTPUT_DIR, "ps4_section12_misspec_boxplots.png")
plt.savefig(fig_path_12, dpi=150, bbox_inches='tight')
plt.close()
print(f"\n[Saved misspecification boxplot: {fig_path_12}]")

# ─────────────────────────────────────────────────────────────────────────────
# FINAL SUMMARY TABLES
# ─────────────────────────────────────────────────────────────────────────────

print("\n\n" + "="*70)
print("  FINAL SUMMARY — Section 1.2 Full Table")
print("="*70)
all_12_labels = [
    "Correctly specified",
    "Only PS misspecified",
    "Only Outcome misspecified",
    "Both misspecified",
]
for label_short, res in zip(all_12_labels, all_12_results):
    m = compute_metrics(res)
    print(f"\n  {label_short}")
    print(f"  {'Estimator':<15} {'Bias':>10} {'Variance':>12} {'MSE':>12}")
    print(f"  {'-'*50}")
    for name in ['IPW', 'Regression', 'AIPW']:
        print(f"  {name:<15} {m[name]['Bias']:>10.5f} {m[name]['Variance']:>12.6f} {m[name]['MSE']:>12.6f}")

print("\n\nDone. Figures saved to:", OUTPUT_DIR)




  SECTION 1.2 — Misspecification (DGP coef = 0.25)

Running simulations for: Only PS misspecified...

  Only PS misspecified
(PS: X1 only, Outcome: X1+X2)
Estimator             Bias     Variance          MSE
--------------------------------------------------
IPW                0.24438     0.015480     0.075187
Regression         0.00201     0.008767     0.008762
AIPW               0.00202     0.008763     0.008758

Running simulations for: Only Outcome misspecified...

  Only Outcome misspecified
(PS: X1+X2, Outcome: X1 only)
Estimator             Bias     Variance          MSE
--------------------------------------------------
IPW                0.00487     0.009073     0.009087
Regression         0.24801     0.017070     0.078561
AIPW               0.00386     0.008530     0.008536

Running simulations for: Both misspecified...

  Both misspecified
(PS: X1 only, Outcome: X1 only)
Estimator             Bias     Variance          MSE
--------------------------------------------------

/tmp/ipykernel_3203/62222847.py:56: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data_per_scenario,
/tmp/ipykernel_3203/62222847.py:56: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data_per_scenario,
/tmp/ipykernel_3203/62222847.py:56: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data_per_scenario,



[Saved misspecification boxplot: /sessions/jolly-blissful-ramanujan/mnt/Econometrics II/ps4_section12_misspec_boxplots.png]


  FINAL SUMMARY — Section 1.2 Full Table

  Correctly specified
  Estimator             Bias     Variance          MSE
  --------------------------------------------------
  IPW                0.00064     0.008583     0.008575
  Regression         0.00021     0.008072     0.008064
  AIPW               0.00027     0.008103     0.008095

  Only PS misspecified
  Estimator             Bias     Variance          MSE
  --------------------------------------------------
  IPW                0.24438     0.015480     0.075187
  Regression         0.00201     0.008767     0.008762
  AIPW               0.00202     0.008763     0.008758

  Only Outcome misspecified
  Estimator             Bias     Variance          MSE
  --------------------------------------------------
  IPW                0.00487     0.009073     0.009087
  Regression         0.24801     0.017070     0

*Only PS misspecified:*
IPW estimator is biased; regression estimator is not. AIPW performs basically identically to the regression estimator.

*Only outcome misspecified:*
The regression estimator is biased; IPW estimator is not. AIPW performs very similarly to the IPW estimator (slightly better, for some reason...).

*Both PS and outcome misspecified:*
Now we're in trouble -- the IPW estimator and the regression estimator are bot biased, and so the AIPW estimator is too. But the fact that it performs well as long as *either* the PS or the outcome is correctly specified demonstrates the double-robustness property of AIPW.